In [2]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:99% !important;}
div.cell.code_cell.rendered{width:100%;}
div.input_prompt{padding:0px;}
div.CodeMirror {font-family:Consolas; font-size:20pt;}
.inner_cell{font-size:20pt;}
div.text_cell_render pre code {font-size:20pt; line-height:30px;}
div.output {font-size:20pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:20pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render ul li{font-size:20pt;padding:5px;}
table.dataframe{font-size:20px;}
</style>
"""))

<font color="red" size="6"><b>ch14. 웹 데이터 수집 I</b></font>

# 1절. BeautifulSoup과 parser
    (정적 웹크롤링, 공공api사용)
    
`pip install bs4` 아나콘다를 설치하면 자동 설치되는 패키지에 포함
- 공식 사이트 : https://www.crummy.com/software/BeautifulSoup/
- documentation : https://www.crummy.com/software/BeautifulSoup/bs4/doc/

In [2]:
import requests # HTTP요청 처리하는 lib
# file:// == c:
# http://www.
from requests_file import FileAdapter

In [12]:
# 로컬에 있는 파일을 웹요청하듯이 읽어오는 작업
s = requests.Session()
s.mount("file://", FileAdapter()) # file://로 시작하는 url을 어댑터가 처리
response = s.get("file:///ai/lecNote/01_python/data/ch14_sample.html") # c:/ai/lecNote/01_python/data/ch14_sample.html
response

<Response [200]>

In [13]:
if response:
    print('해당 url에 접근함')
else:
    print('해당 url에 거부됨')

해당 url에 접근함


In [14]:
response.status_code
# 200 : 정상
# 404 : 없는 페이지

200

In [15]:
response.content # html의 바이너리 형식의 내용

b'<!DOCTYPE html>\r\n<html lang="en">\r\n<head>\r\n  <meta charset="UTF-8">\r\n</head>\r\n<body>\r\n  <h1 class="greeting css" id="text">Hello, CSS</h1>\r\n  <h1 class="css">Hi, CSS</h1>\r\n  <div id="subject">subject \xec\x84\xa0\xed\x83\x9d\xec\x9e\x90 \xec\x95\x88\xec\x9d\x98 \xeb\x82\xb4\xec\x9a\xa9</div>\r\n  <p>CSS \xec\x84\xa0\xed\x83\x9d\xec\x9e\x90\xeb\x8a\x94 \xeb\x8b\xa4\xec\x96\x91\xed\x95\x9c \xea\xb3\xb3\xec\x97\x90\xec\x84\x9c \xed\x99\x9c\xec\x9a\xa9\xeb\x90\xa9\xeb\x8b\x88\xeb\x8b\xa4</p>\r\n  <div class="contents">\r\n    \xec\x84\xa0\xed\x83\x9d\xec\x9e\x90\xeb\xa5\xbc \xec\x96\xb4\xeb\x96\xbb\xea\xb2\x8c \xec\x9e\x91\xec\x84\xb1\xed\x95\x98\xeb\x8a\x90\xeb\x83\x90\xec\x97\x90 \xeb\x94\xb0\xeb\x9d\xbc\r\n    <span>\xeb\x8b\xa4\xeb\xa5\xb8<b>\xec\x9a\x94\xec\x86\x8c\xea\xb0\x80 \xeb\xb0\x98\xed\x99\x98</b></span>\xeb\x90\xa9\xeb\x8b\x88\xeb\x8b\xa4\r\n  </div>\r\n  <div>CSS \xec\x84\xa0\xed\x83\x9d\xec\x9e\x90\xeb\x8a\x94 \xeb\x8b\xa4\xec\x96\x91\xed\x95\x9c \xea\xb3\

In [16]:
print(response.content.decode('utf-8'))

<!DOCTYPE html>
<html lang="en">
<head>
  <meta charset="UTF-8">
</head>
<body>
  <h1 class="greeting css" id="text">Hello, CSS</h1>
  <h1 class="css">Hi, CSS</h1>
  <div id="subject">subject 선택자 안의 내용</div>
  <p>CSS 선택자는 다양한 곳에서 활용됩니다</p>
  <div class="contents">
    선택자를 어떻게 작성하느냐에 따라
    <span>다른<b>요소가 반환</b></span>됩니다
  </div>
  <div>CSS 선택자는 다양한 곳에 <b>활용</b>됩니다</div>
</body>
</html>


In [19]:
response.text

'<!DOCTYPE html>\r\n<html lang="en">\r\n<head>\r\n  <meta charset="UTF-8">\r\n</head>\r\n<body>\r\n  <h1 class="greeting css" id="text">Hello, CSS</h1>\r\n  <h1 class="css">Hi, CSS</h1>\r\n  <div id="subject">subject 선택자 안의 내용</div>\r\n  <p>CSS 선택자는 다양한 곳에서 활용됩니다</p>\r\n  <div class="contents">\r\n    선택자를 어떻게 작성하느냐에 따라\r\n    <span>다른<b>요소가 반환</b></span>됩니다\r\n  </div>\r\n  <div>CSS 선택자는 다양한 곳에 <b>활용</b>됩니다</div>\r\n</body>\r\n</html>'

In [24]:
# html 파싱 객체
from bs4 import BeautifulSoup
soup = BeautifulSoup(response.text, #response.content, 
                    "html.parser")
# soup

In [37]:
# 1. soup.select_one('선택자') : 해당 선택자 처음 하나 엘리먼트만 
el = soup.select_one('h1.css')
print('el =>', el)
print('el.text   =>', el.text)
print('el.string =>', el.string)
print('el의 속성들 =>', el.attrs)
print('el의 class속성 =>', el.attrs['class'])
print('el의 class속성 =>', el.attrs.get('class'))
# print('el의 href속성(없는 속성은 에러) =>', el.attrs.get['href'])
print('el의 href속성 =>', el.attrs.get('href'))
print('el의 name =>', el.name)

el => <h1 class="greeting css" id="text">Hello, CSS</h1>
el.text   => Hello, CSS
el.string => Hello, CSS
el의 속성들 => {'class': ['greeting', 'css'], 'id': 'text'}
el의 class속성 => ['greeting', 'css']
el의 class속성 => ['greeting', 'css']
el의 href속성 => None
el의 name => h1


In [45]:
# 2. soup.select('선택자') : 해당 선택자 엘리먼트 다 list로
els = soup.select('h1.css')
print('els =>', els)
print('els들의 text =>', [el.text for el in els])
print('els들의 string =>', [el.string for el in els])
print('els들의 속성들 =>', [el.attrs for el in els])
print('els들의 class 속성 =>', [el.attrs.get('class') for el in els])

els => [<h1 class="greeting css" id="text">Hello, CSS</h1>, <h1 class="css">Hi, CSS</h1>]
els들의 text => ['Hello, CSS', 'Hi, CSS']
els들의 string => ['Hello, CSS', 'Hi, CSS']
els들의 속성들 => [{'class': ['greeting', 'css'], 'id': 'text'}, {'class': ['css']}]
els들의 class 속성 => [['greeting', 'css'], ['css']]


In [50]:
# 3. soup.find(태그, 속성)  vs soup.select_one('선택자') : 해당 속성을 갖은 태그 처음 하나만 
print('select_one :', soup.select_one('h1.css'))
print('find       :', soup.find('h1', {'class':'css'}))
print('find       :', soup.find('h1', class_='css'))
print()
print('select_one :', soup.select_one('h1#text'))
print('select_one :', soup.find('h1', {'id':'text'}))

select_one : <h1 class="greeting css" id="text">Hello, CSS</h1>
find       : <h1 class="greeting css" id="text">Hello, CSS</h1>
find       : <h1 class="greeting css" id="text">Hello, CSS</h1>

select_one : <h1 class="greeting css" id="text">Hello, CSS</h1>
select_one : <h1 class="greeting css" id="text">Hello, CSS</h1>


In [61]:
# 4. soup.find_all(태그, 속성) vs. soup.select('선택자') : 해당 엘리먼트 다 list로
print('모든 h1.css와 span태그 :', soup.select('h1.css, span'))
print('모든 h1.css와 span태그 :', soup.find_all(['h1'], class_='css') +
                                soup.find_all('span'))

모든 h1.css와 span태그 : [<h1 class="greeting css" id="text">Hello, CSS</h1>, <h1 class="css">Hi, CSS</h1>, <span>다른<b>요소가 반환</b></span>]
모든 h1.css와 span태그 : [<h1 class="greeting css" id="text">Hello, CSS</h1>, <h1 class="css">Hi, CSS</h1>, <span>다른<b>요소가 반환</b></span>]


In [71]:
# 없는 엘리먼트 찾기
print('find_all(빈list) :', soup.find_all('a'))
print('find(None)       :', soup.find('a'))
print('select(빈list) :', soup.select('a'))
print('select_one(None) :', soup.select_one('a'))

find_all(빈list) : []
find(None)       : None
select(빈list) : []
select_one(None) : None


# 2절. 정적 웹 데이터 수집(정적 웹크롤링)
## 2.1 BeautifulSoup 모듈을 활용한 html 웹 데이터 수집
### 1) 환율정보 가져오기(네이버증권 > 시장지표)

- https://finance.naver.com/marketindex/

    * 크롤링 허용범위는 사이트마다 ~/robots.txt에서 확인할 수 있음
        - Allow : 크롤링 허용가능한 폴더
        - Disallow : 크롤링 제한 폴더

In [78]:
# soup객체 생성 방법1
import requests
from bs4 import BeautifulSoup
url = 'https://finance.naver.com/marketindex/'
response = requests.get(url)
# response # Response 
print(response.status_code)
# response.text # response.content
soup = BeautifulSoup(response.text, 'html.parser')

200


In [86]:
# soup객체 생성 방법2
from urllib.request import urlopen
url = 'https://finance.naver.com/marketindex/'
response = urlopen(url)
# response # HTTPResponse
print(response.status)
# print(response.read().decode('cp949'))
soup = BeautifulSoup(response, 'html.parser')

200


In [100]:
p = '1,417,000.70'
float(p.replace(',','')) # 방법1
float(''.join(p.split(','))) # 방법2

1417000.7

In [104]:
# div.head_info 밑의 span.value (find계열)
prices = []
headinfos = soup.find_all('div', class_='head_info')
for headinfo in headinfos:
    # print(headinfo)
    price = headinfo.find('span', class_='value')
    prices.append(float(''.join(price.text.split(','))))
print(prices)

[1417.7, 889.17, 1635.6, 210.09, 159.24, 1.1541, 1.3507, 99.71, 83.2, 1863.92, 4441.1, 200855.05]


In [109]:
# span.value (find계열)
price_els = soup.find_all('span', class_='value')
prices = [round(float(price.text.replace(',','')), 1) for price in price_els]
print(prices)

[1417.7, 889.2, 1635.6, 210.1, 159.2, 1.2, 1.4, 99.7, 83.2, 1863.9, 4441.1, 200855.0]


In [117]:
# 금액들 : div.head_info 밑의 span.value 
price_els = soup.select('div.head_info > span.value')
len(price_els)

12

In [116]:
# 타이틀
title_els = soup.select('h3.h_lst > span.blind')
len(title_els)

12

In [121]:
# 단위들 : div.head_info > span > span.blind
unit_els = soup.select('div.head_info > span > span.blind')
len(unit_els)
units = [unit_el.string for unit_el in unit_els]
units.insert(7, '')
units

['원', '원', '원', '원', '엔', '달러', '달러', '', '달러', '원', '달러', '원']

In [124]:
# 상승/하락 :  div.head_info > span.blind
trend_els = soup.select('div.head_info > span.blind')
#trend_els

In [125]:
len(title_els), len(price_els), len(units), len(trend_els)

(12, 12, 12, 12)

In [127]:
for idx in range(len(title_els)):
    print("{} : {} {} - {}".format(title_els[idx].text,
                                  price_els[idx].text,
                                  units[idx],
                                  trend_els[idx].text))

미국 USD : 1,417.70 원 - 상승
일본 JPY(100엔) : 889.17 원 - 상승
유럽연합 EUR : 1,635.60 원 - 상승
중국 CNY : 210.09 원 - 상승
달러/일본 엔 : 159.2400 엔 - 상승
유로/달러 : 1.1541 달러 - 하락
영국 파운드/달러 : 1.3507 달러 - 하락
달러인덱스 : 99.7100  - 상승
WTI : 83.2 달러 - 상승
휘발유 : 1863.92 원 - 하락
국제 금 : 4441.1 달러 - 상승
국내 금 : 200855.05 원 - 상승


In [128]:
for title, price, unit, trend in zip(title_els, price_els, units, trend_els):
    print("{} : {}{} - {}".format(title.text, price.text, unit, trend.text))

미국 USD : 1,417.70원 - 상승
일본 JPY(100엔) : 889.17원 - 상승
유럽연합 EUR : 1,635.60원 - 상승
중국 CNY : 210.09원 - 상승
달러/일본 엔 : 159.2400엔 - 상승
유로/달러 : 1.1541달러 - 하락
영국 파운드/달러 : 1.3507달러 - 하락
달러인덱스 : 99.7100 - 상승
WTI : 83.2달러 - 상승
휘발유 : 1863.92원 - 하락
국제 금 : 4441.1달러 - 상승
국내 금 : 200855.05원 - 상승


In [132]:
import pandas as pd
data = []
for title, price, unit, trend in zip(title_els, price_els, units, trend_els):
    data.append({'title' : title.text,
                 'price' : float(price.text.replace(',','')),
                 'unit' : unit,
                 'trend' : trend.text})
pd.DataFrame(data)#.to_csv('data/file.csv', index=False)

,title,price,unit,trend
0,미국 USD,1417.7000,원,상승
1,일본 JPY(100엔),889.1700,원,상승
2,유럽연합 EUR,1635.6000,원,상승
3,중국 CNY,210.0900,원,상승
4,달러/일본 엔,159.2400,엔,상승
5,유로/달러,1.1541,달러,하락
6,영국 파운드/달러,1.3507,달러,하락
7,달러인덱스,99.7100,,상승
8,WTI,83.2000,달러,상승
9,휘발유,1863.9200,원,하락


In [135]:
data = []
for title, price, unit, trend in zip(title_els, price_els, units, trend_els):
    data.append([title.text, float(price.text.replace(',','')), unit, trend.text])
pd.DataFrame(data, columns=['title','price','unit','trend'])

,title,price,unit,trend
0,미국 USD,1417.7000,원,상승
1,일본 JPY(100엔),889.1700,원,상승
2,유럽연합 EUR,1635.6000,원,상승
3,중국 CNY,210.0900,원,상승
4,달러/일본 엔,159.2400,엔,상승
5,유로/달러,1.1541,달러,하락
6,영국 파운드/달러,1.3507,달러,하락
7,달러인덱스,99.7100,,상승
8,WTI,83.2000,달러,상승
9,휘발유,1863.9200,원,하락


### 2) 이번주 로또번호 출력
- 방법2에서 User-Agent를 추가하여 soup생성
- https://search.daum.net/search?nil_suggest=btn&w=tot&DA=SBC&q=lotto (다음에서 lotto검색)
```
    1236회(2026.08.08 추첨)
    당첨번호 [12, 18, 21, 29, 34, 38]
    보너스 10
```

In [146]:
# 방법1
import requests
from bs4 import BeautifulSoup
url = 'https://search.daum.net/search?nil_suggest=btn&w=tot&DA=SBC&q=lotto'
response = requests.get(url)
print('response의 상태 :',response.status_code)
soup = BeautifulSoup(response.text, 'html.parser')
# soup

response의 상태 : 200


In [151]:
# 방법2
from urllib.request import urlopen, Request
headers = {'User-Agent':
          'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/151.0.0.0 Safari/537.36'}
request = Request(url, headers=headers)

response = urlopen(request)
print('response의 상태 :', response.status)
soup = BeautifulSoup(response, 'html.parser')
#soup

response의 상태 : 200


In [176]:
# 1236회 (2026.08.08 추첨)
# 당첨번호 [12, 18, 21, 29, 34, 38]
# 보너스 10
times = soup.select_one('div.prize span.f_red').text
date = soup.select_one('div.prize > span.date').text
title1 = soup.select_one('div.prize > strong').text[-4:]
lotto_numbers = soup.select('div.lottonum > span.ball:nth-last-child(n+3)')
lotto_numbers = soup.select('div.lottonum > span.ball')[:-2]
title2 = soup.select_one('div.lottonum span.screen_out').text
bonus_number = soup.select_one('div.lottonum > span.bg_ball1').text
print(times, date)
print(title1, [int(numbers.text) for numbers in lotto_numbers])
print(title2, bonus_number)

1236회 (2026.08.08 추첨)
당첨번호 [12, 18, 21, 29, 34, 38]
보너스 10


In [193]:
# 위의 select계열 함수를 find계열함수로 변경하여 구현해 보기
# times = soup.select_one('div.prize span.f_red').text
prize = soup.find('div', class_='prize')
times = prize.find('span', class_='f_red').text

# date = soup.select_one('div.prize > span.date').text
date = prize.find('span', class_='date').text

# title1 = soup.select_one('div.prize > strong').text[-4:]
title1 = prize.find('strong').text[-4:]

# lotto_numbers = soup.select('div.lottonum > span.ball')[:-2]
lottonum = soup.find('div', class_='lottonum')
lotto_numbers = lottonum.find_all('span', class_='ball')[:-2]

# title2 = soup.select_one('div.lottonum span.screen_out').text
title2 = lottonum.find('span', class_='screen_out').text

# bonus_number = soup.select_one('div.lottonum > span.bg_ball1').text
bonus_number = lottonum.find('span', class_='bg_ball1').text

print(times, date)
print(title1, [int(numbers.text) for numbers in lotto_numbers])
print(title2, bonus_number)

1236회 (2026.08.08 추첨)
당첨번호 [12, 18, 21, 29, 34, 38]
보너스 10


### 3) 다음 뉴스 검색 리스트
```
no title   href
0  타이틀1  http://~
1  타이틀2  http://~
2  타이틀3  http://~
```

In [2]:
# 방법1
import requests
from bs4 import BeautifulSoup
word = '개미들'
url = f'https://search.daum.net/search?w=news&q={word}&enc=utf8&cluster=y&cluster_page=1&DA=DNS'
print(url)
response = requests.get(url)
print(response.status_code)
soup = BeautifulSoup(response.text, 'html.parser')

https://search.daum.net/search?w=news&q=개미들&enc=utf8&cluster=y&cluster_page=1&DA=DNS
200


In [213]:
# 방법2
from urllib.request import urlopen, Request
from urllib.parse import quote
word = quote('비트코인')
url = f'https://search.daum.net/search?w=news&q={word}&enc=utf8&cluster=y&cluster_page=1&DA=DNS'
print(url)
headers = {'User-Agent':
          'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/151.0.0.0 Safari/537.36'}
# request = Request(url, headers=headers)
request = Request(url)
request.add_header('User-Agent', 
                  'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/151.0.0.0 Safari/537.36')
response = urlopen(request)
print(response.status)
soup = BeautifulSoup(response, 'html.parser')
# soup

https://search.daum.net/search?w=news&q=%EB%B9%84%ED%8A%B8%EC%BD%94%EC%9D%B8&p=1
200


In [3]:
items_find_list = [] # 검색한 결과를 담을 dict 리스트
items_el = soup.select('div.item-title > strong.tit-g > a')
for idx, item in enumerate(items_el):
    # print(idx, item)
    items_find_list.append({'no':idx,
                            'title':item.text,
                            'link':item.attrs.get('href')})
import pandas as pd
pd.DataFrame(items_find_list)

,no,title,link
0,0,"""불기둥 무서워""…'천스닥' 전망에도 개미들 '하락 베팅'",http://v.daum.net/v/20260811163537806
1,1,서학개미들 'SK하닉' 아닌 여기 몰렸다,http://v.daum.net/v/20260808095848252
2,2,"“주식 잃고 대출도 막혔다”…600조 날린 개미들, 삼전닉스에 다시 희망을?[주형...",http://v.daum.net/v/20260808055128543
3,3,국장 뜨는 개미들과 딴판…‘큰 손’ 블랙록이 담은 종목은,http://v.daum.net/v/20260812093738363
4,4,장기투자 하라더니 정책은 오락가락…개미들 다시 美 증시로,http://v.daum.net/v/20260810151107970
5,5,"“고점 물린 레버리지 ETF 개미들, 本株·일반 ETF로 갈아타라”",http://v.daum.net/v/20260811003738083
6,6,'200조 쏜대' 개미들 환호…일본서 또 '사상 최고' 배당 뜬다,http://v.daum.net/v/20260813073709577
7,7,"개미들 이달 '코스피 베팅', 외국인과 정반대 흐름",http://v.daum.net/v/20260813082250549
8,8,개미들 증시에 질렸다?…투자 예탁금 100조 붕괴 코앞,http://v.daum.net/v/20260805114326291
9,9,"""내 주식 휴지조각 되나""…상폐 공포에 떠는 개미들 '발 동동'",http://v.daum.net/v/20260813075209804


In [227]:
items_find_list = [] # 검색한 결과를 담을 2차원 리스트
items_el = soup.select('div.item-title > strong.tit-g > a')
for idx, item in enumerate(items_el):
    items_find_list.append([idx, item.text, item.attrs.get('href')])
pd.DataFrame(items_find_list, columns=['순번','기사제목','링크'])

,순번,기사제목,링크
0,0,9000만원 깨진 비트코인…美물가지수 앞두고 위험회피 심리↑,http://v.daum.net/v/20260812104759348
1,1,[코인뉴스] 비트코인 박스권 지속…다음 변수는?,http://v.daum.net/v/20260812093050056
2,2,[코인뉴스] CPI 앞둔 비트코인…박스권 속 엇갈린 베팅,http://v.daum.net/v/20260812163039451
3,3,9000만원대 갇힌 비트코인…'100만弗 vs 4만弗' 극단 전망,http://v.daum.net/v/20260812161211645
4,4,[코인시세] 비트코인 6만3천달러대 약세…CPI 경계감 지속,http://v.daum.net/v/20260812103841887
5,5,"[08:03 가상자산] 비트코인, 美 CPI 결과 발표 앞두고 9000만원 선 붕괴",http://v.daum.net/v/20260812080815138
6,6,"트럼프 회사, 비트코인에 3300억 베팅했다 '쓴맛'⋯손실 12배 늘었다",http://v.daum.net/v/20260811144238992
7,7,"금리 공포 걷히자 ""금값 조정 끝났다""… 비트코인은 나 홀로 약세",http://v.daum.net/v/20260812164709172
8,8,[아주경제 코이너스 브리핑] 호르무즈 불확실성에…비트코인 6만3000달러대 횡보,http://v.daum.net/v/20260812082712570
9,9,비트코인 다시 6.3만달러대로…美 CPI 발표 ‘촉각’ [코인 모닝콜],http://v.daum.net/v/20260812082137419


In [4]:
# 다음 뉴스 검색 함수(원하는 키워드, 원하는 페이지로)
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
def collect_list(keyword, page):
    'keyword로 해당 page에 검색한 결과 dict list return'
    # url = f'https://search.daum.net/search?w=news&q={keyword}&enc=utf8&cluster=y&cluster_page=1&DA=PGD&p={page}'
    url = 'https://search.daum.net/search?w=news&enc=utf8&cluster=y&cluster_page=1&DA=PGD'
    params = {'q':keyword, 'p':page}
    response = requests.get(url, params=params)
    soup = BeautifulSoup(response.text, 'html.parser')
    items_find_list = []
    items_el = soup.select('div.item-title > strong.tit-g > a')
    for idx, item in enumerate(items_el):
        items_find_list.append({'no':(page-1)*10 + idx, 
                                'title':item.text,
                                'link':item.attrs.get('href')})
    return items_find_list

In [5]:
collect_list('태풍',2)

[{'no': 10,
  'title': ' 블랙핑크 지수 ‘청바지에 흰티면 OK!’[포토엔HD] ',
  'link': 'http://v.daum.net/v/20260811003242018'},
 {'no': 11,
  'title': ' 송혜교, 흰티에 청바지 일상 사진 공개하자..절친 다 몰려왔다 ',
  'link': 'http://v.daum.net/v/20260806151007109'},
 {'no': 12,
  'title': ' 흰 티에 청바지…송혜교, 폭염 속에도 빛나는 비주얼 [N샷] ',
  'link': 'http://v.daum.net/v/20260806151445411'},
 {'no': 13,
  'title': ' ‘똥배 논란’ 사라지더니…홍진영, 오조오억년 만에 치마 벗고 청바지 ',
  'link': 'http://v.daum.net/v/20260722074501440'},
 {'no': 14,
  'title': ' 올가을 청바지는 더 이상 기본템이 아닙니다 ',
  'link': 'http://v.daum.net/v/20260810234410440'},
 {'no': 15,
  'title': ' "지퍼 올려!" 엄마들 질색한 청바지…Z세대는 열광한다는데 ',
  'link': 'http://v.daum.net/v/20260724132009295'},
 {'no': 16,
  'title': ' 여름에도 교복처럼 입을 청바지 발견했습니다 ',
  'link': 'http://v.daum.net/v/20260722183437854'},
 {'no': 17,
  'title': ' 돌싱 박지윤, 8㎏ 빼고 또 다이어트 하더니 청바지핏 완벽 ',
  'link': 'http://v.daum.net/v/20260724152711935'},
 {'no': 18,
  'title': ' 여름과 가장 잘 어울리는 청바지는 따로 있습니다 ',
  'link': 'http://v.daum.net/v/20260724180507627'},
 {

In [11]:
r = []
r.extend([1, 2, 3])
r.extend([4, 5, 6])
r

[1, 2, 3, 4, 5, 6]

In [12]:
result = [] # 해당 키워드 원하는 페이지 수만큼 검색한 결과를 담을 변수 dict 리스트
pages = 3
for page in range(1, pages+1):
    print(f'== {page} 페이지 수집 중 ==')
    item_result = collect_list('추석열차', page)
    result.extend(item_result)
    time.sleep(3)
pd.DataFrame(result)

== 1 페이지 수집 중 ==
== 2 페이지 수집 중 ==
== 3 페이지 수집 중 ==


,no,title,link
0,0,"추석 열차표 예매, KTX·SRT 따로 안 해도 된다…9월 1일 통합 앱 출시",http://v.daum.net/v/20260802115907094
1,1,“올해 추석부터 열차값 10% 아끼세요”…KTX·SRT 9월 통합,http://v.daum.net/v/20260802192100597
2,2,"나중엔 늦는다…추석·10월에 3일 연차로 9일 황금연휴 완성, 어디에 걸까? [여...",http://v.daum.net/v/20260807123245162
3,3,'올해도 포기했는데' 설렌다…추석 앞두고 벌어진 대반전,http://v.daum.net/v/20260812190246138
4,4,"""지금 안 사면 늦는다""…직장인들 난리 난 황금연휴, 항공권·숙소 예약전쟁",http://v.daum.net/v/20260812105728812
5,5,"[현장]코레일·SR 통합…추석, 통합운영 시험대",http://v.daum.net/v/20260803145005360
6,6,"'대국민 티켓팅' 경쟁률 낮아질까…'최대 17,000석' 는다",http://v.daum.net/v/20260813071804287
7,7,KTX·SRT 9월부터 통합…부산·울산 추석 귀성표 전쟁 숨통 트이나,http://v.daum.net/v/20260805123449011
8,8,"코레일-SRT, 철도 회원 통합 시작…""9월 운행 열차부터 통합 예매""",http://v.daum.net/v/20260713171250054
9,9,"철도, 좌석 주간 11만 석 확대 및 예매 앱 단일화",http://v.daum.net/v/20260812202647188


In [16]:
keywords = ['김혜수', '영화']
pages = 3
result0 = [] # keyword[0] 1~pages페이지까지 검색한 결과 dict list
result1 = [] # keyword[1] 1~pages페이지까지 검색한 결과 dict list
for i, keyword in enumerate(keywords):
    print(f'= = {i+1}번째 검색어 {keyword} 검색 결과 수집({pages}페이지) 중입니다 = =')
    for page in range(1, pages+1):
        if i==0:
            result0.extend(collect_list(keyword, page))
        else:
            result1.extend(collect_list(keyword, page))
        time.sleep(3)

= = 1번째 검색어 김혜수 검색 결과 수집(3페이지) 중입니다 = =
= = 2번째 검색어 영화 검색 결과 수집(3페이지) 중입니다 = =


In [17]:
result0_df = pd.DataFrame(result0)
result1_df = pd.DataFrame(result1)
result0_df.sample()

,no,title,link
3,3,‘지금 불륜’ 김혜수 55세 맞아? 감탄 부르는 비주얼,http://v.daum.net/v/20260803172404089


In [18]:
result1_df.head()

,no,title,link
0,0,여름 대작 피했더니 9월에 6편 몰린 한국영화… ‘공멸’ 잔혹사 끊을까 [영화 뷰],http://v.daum.net/v/20260812082840601
1,1,"[단독] 염정아, 스릴러 영화 '투피스'서 문가영과 만난다! 꿈의 조합 완성",http://v.daum.net/v/20260813065649908
2,2,"'오디세이' 봤다면, 이 영화도 함께 보세요",http://v.daum.net/v/20260811160736520
3,3,"다대포 바다, 영화로 물든다…제4회 다대포선셋영화축제 14일 개막",http://v.daum.net/v/20260813101444334
4,4,"“우리 증조부도 매국노, 조상 업보 청산중”…영화 ‘암살’ 이경영 후손의 고백",http://v.daum.net/v/20260813094909942


In [20]:
result0_df.to_csv(f'data/ch14_{keywords[0]}.csv', index=False, encoding='cp949')
result1_df.to_csv(f'data/ch14_{keywords[1]}.csv', index=False, encoding='cp949')

### 4) User-Agent를 추가하여 크롤링
- request.get(url), urlopen(url)함수를 사용하면 크롤링이 막혀있는 사이트
- 방법2에서 User-Agent를 추가혀여 크롤링

- https://www.melon.com/robots.txt 에서 일부 경로는 User-Agent에 봇이 지정

In [24]:
# 방법1
import requests
from bs4 import BeautifulSoup
url = 'https://www.melon.com/chart/'
melonResponse = requests.get(url)
print(melonResponse.status_code)
soup = BeautifulSoup(melonResponse.text, 'html.parser')
soup

406


In [26]:
# 방법2
from urllib.request import urlopen
from bs4 import BeautifulSoup
url = 'https://www.melon.com/chart/'
#melonResponse = urlopen(url) # HTTPError: HTTP Error 406: Not Acceptable

In [30]:
# User-Agent를 추가하여 방법2
from urllib.request import urlopen, Request
from bs4 import BeautifulSoup
url = 'https://www.melon.com/chart/'
headers = {'user-agent':
          'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/151.0.0.0 Safari/537.36'}
# melonpage = Request(url, headers=headers)
melonpage = Request(url)
melonpage.add_header('user-agent',
                    'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/151.0.0.0 Safari/537.36')
melonResponse = urlopen(melonpage)
print(melonResponse.status)
soup = BeautifulSoup(melonResponse, 'html.parser')
# soup

200


In [35]:
# User-Agent를 추가하여 방법1
import requests
from bs4 import BeautifulSoup
url = 'https://www.melon.com/chart/'
headers = {'user-agent':
          'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/151.0.0.0 Safari/537.36'}
melonResponse = requests.get(url, headers=headers)
print(melonResponse.status_code)
soup = BeautifulSoup(melonResponse.text, 'html.parser')
#soup

200


In [76]:
# 1위 : LOVE ATTACK | RESCENE (리센느)의 url {}
# 순위, 곡명, 가수, 가수페이지
rank_els = soup.select('div.wrap.t_center > span.rank')[1:] # 맨앞엔 '순위'
ranks = [rank.text for rank in rank_els]

title_els = soup.select('div.ellipsis.rank01 > span > a')
titles = [title.text for title in title_els]

singer_els = soup.select('span.checkEllipsis') # 40위, 90위 가수가 복수명
singers = [singer.text.replace('\xa0','') for singer in singer_els]

links = []
for singer_el in singer_els:
    # print(singer_el)
    singer_link = 'https://www.melon.com' + singer_el.find('a').attrs.get('href')
    links.append(singer_link)
len(ranks), len(titles), len(singers), len(links)

# for idx, (title, singer, link) in enumerate(zip(titles, singers, links)):
#     print("{}위. {} | {}".format(idx+1, title, singer))
melon_chat_list = []
for rank, title, singer, link in zip(ranks, titles, singers, links):
    # print(f'{rank}위 {title} | {singer}')
    melon_chat_list.append({
        '순위':rank,
        '곡명':title,
        '가수':singer,
        '가수페이지':link
    })
pd.DataFrame(melon_chat_list) # pd.options.display.max_rows(60)행이상은 중간이 생략

,순위,곡명,가수,가수페이지
0,1,LOVE ATTACK,RESCENE (리센느),https://www.melon.com/artist/detail.htm?artist...
1,2,갑자기,아이오아이 (I.O.I),https://www.melon.com/artist/detail.htm?artist...
2,3,REDRED,CORTIS (코르티스),https://www.melon.com/artist/detail.htm?artist...
3,4,LEMONADE,aespa,https://www.melon.com/artist/detail.htm?artist...
4,5,Pretty Girl,RESCENE (리센느),https://www.melon.com/artist/detail.htm?artist...
...,...,...,...,...
95,96,BLACKHOLE,IVE (아이브),https://www.melon.com/artist/detail.htm?artist...
96,97,FOCUS,Hearts2Hearts (하츠투하츠),https://www.melon.com/artist/detail.htm?artist...
97,98,Soda Pop,"KPop Demon Hunters Cast, Danny Chung, Saja Boy...",https://www.melon.com/artist/detail.htm?artist...
98,99,OVERDRIVE,TWS (투어스),https://www.melon.com/artist/detail.htm?artist...


### 5) 네이버 지식인 검색(open API 사용X)
- 특정 keyword를 특정 페이지 수만큼

In [78]:
# 방법1
from requests import get
from bs4 import BeautifulSoup
keyword = '쳇지피티'
url = f'https://kin.naver.com/search/list.naver?query={keyword}'
print(url)
response = get(url)
print(response.status_code)
soup = BeautifulSoup(response.text, 'html.parser')

https://kin.naver.com/search/list.naver?query=쳇지피티
200


In [83]:
# 방법2
from urllib.request import urlopen
from bs4 import BeautifulSoup
from urllib.parse import quote
keyword = quote('쳇지피티')
url = f'https://kin.naver.com/search/list.naver?query={keyword}'
print(url)
response = urlopen(url)
print(response.status)
soup = BeautifulSoup(response, 'html.parser')

https://kin.naver.com/search/list.naver?query=%EC%B3%87%EC%A7%80%ED%94%BC%ED%8B%B0
200


In [99]:
# keyword를 원하는 페이지 수만큼
# 방법1
from requests import get
from bs4 import BeautifulSoup
keyword = '쳇지피티'
pages = 2
items_list = [] # 크롤링한 데이터를 담을 list
for page in range(1, pages+1):
    url = f'https://kin.naver.com/search/list.naver?query={keyword}&page={page}'
    # print(url)
    # url = 'https://kin.naver.com/search/list.naver'
    # params = {'query':keyword, 'page':page}
    # response = get(url, params=params)
    response = get(url)
    # print(response.status_code)
    soup = BeautifulSoup(response.text, 'html.parser')
    # 글제목, link
    dt_els = soup.find_all('dt')
    for dt_el in dt_els:
        item = dt_el.find('a')
        items_list.append({
            'title':item.text,
            'link':item.attrs.get('href')
        })
df = pd.DataFrame(items_list)

In [102]:
df.head()

,title,link
0,챗지피티 플러스 오류,https://kin.naver.com/qna/detail.naver?dirId=4...
1,ChatGPT 챗지피티 답이 늦는 경우에는 어떻게 ....,https://kin.naver.com/qna/detail.naver?dirId=1...
2,챗gpt Auto,https://kin.naver.com/qna/detail.naver?dirId=1...
3,"챗지피티, 제미나이 음성 사용 용량은 무한대....",https://kin.naver.com/qna/detail.naver?dirId=4...
4,챗지피티 답변,https://kin.naver.com/qna/detail.naver?dirId=8...


## 2.2 open API사용 : json 웹데이터 수집
### 1) 네이버 지식으로 검색 (open API사용 o)

- 네이버개발자센터에서 애플리케이션을 등록(id, pw) => NAVER API HUB

In [4]:
%pip install dotenv

In [5]:
# 환경변수를 쓰기 위한 패키지 : dotenv
from dotenv import load_dotenv
import os
load_dotenv(
    #dotenv_path='.env'
)
print(os.getenv('CLIENT_ID')[:3])
print(os.getenv('CLIENT_SECRET')[:3])

w5G
XZv


In [25]:
# 방법2
import os
import sys
import urllib.request
import json
import pandas as pd
client_id = os.getenv('CLIENT_ID')
client_secret = os.getenv('CLIENT_SECRET')
encText = urllib.parse.quote("쳇지피티")
url = f"https://openapi.naver.com/v1/search/kin.json?query={encText}" # JSON 결과
# request = urllib.request.Request(url)
# request.add_header("X-Naver-Client-Id",client_id)
# request.add_header("X-Naver-Client-Secret",client_secret)
headers = {
    'X-Naver-Client-Id': client_id,
    'X-Naver-Client-Secret':client_secret
}
request = urllib.request.Request(url, headers=headers)
response = urllib.request.urlopen(request)
rescode = response.getcode()
if(rescode==200):
    response_body = response.read()
    # print(response_body.decode('utf-8')[:30])
else:
    print("Error Code:" + rescode)

data = json.loads(response_body) # 문자를 json형태로 변환
print('data의 응답들 :', data.keys())
print('검색결과 갯수 :', len(data['items']))

items = data['items']
items_list = []
for item in items:
    #print(item)
    title = item['title'].replace('<b>','').replace('</b>','')
    link  = item['link']
    description = item['description'].replace('<b>','').replace('</b>','')
    items_list.append([title, link, description])
pd.DataFrame(items_list, columns=['title','link','description'])

data의 응답들 : dict_keys(['lastBuildDate', 'total', 'start', 'display', 'items'])
검색결과 갯수 : 10


,title,link,description
0,박사논문을 잘 쓰려면 쳇지피티 외 어떤ai하나더 추천해줘요,https://kin.naver.com/qna/detail.naver?dirId=4...,박사논문을 잘 쓰려면 쳇지피티 외 어떤ai하나더 추천해줘요. 논문제목 작성 및 논문...
1,쳇지피티 오류 나만 그래요?,https://kin.naver.com/qna/detail.naver?dirId=1...,짜증나네 삭제해도 안되고 왜이러는거에요 현재 쳇지피티 오류 나는거 모두 다 동일한 ...
2,쳇지피티로 사주 해석을 해봤는데 맞는건가요?,https://kin.naver.com/qna/detail.naver?dirId=3...,쳇지피티로 배우자가 어떤 사람인지 사주 해석을 해봤는데 맞는건가요??? 전체 흐름을...
3,박사논문을 잘 쓰려면 쳇지피티 외 어떤ai하나더 추천해줘요,https://kin.naver.com/qna/detail.naver?dirId=4...,박사논문을 잘 쓰려면 쳇지피티 외 어떤ai하나더 추천해줘요. 논문제목 작성 및 논문...
4,고등학교 수행평가에 쳇지피티 쓰면 안되나요?,https://kin.naver.com/qna/detail.naver?dirId=1...,쳇지피티로 쓰던데 쓰면 안되아요? 요즘 지피티 검사기도 있고 선생님들도 지피티쓴 글...
5,카카오톡 쳇지피티,https://kin.naver.com/qna/detail.naver?dirId=1...,이거 사용하는거 무료인가요 ? 돈 나가요!
6,쳇지피티 오류,https://kin.naver.com/qna/detail.naver?dirId=1...,이거 왜이런건가요…? 한번에 너무 많은 요구를 했거나 해당 버전을 동시간대에 사용하...
7,뉴스 보다가 범죄자가 검색한 쳇 지피티 도 조회 한다고 하....,https://kin.naver.com/qna/detail.naver?dirId=6...,제미나이 이런것도 포렌식 해서 다 검열을 하나요 첨 알아서요 검색 목록만 보는줄 알...
8,쳇 지피티 운세 정확한가요?,https://kin.naver.com/qna/detail.naver?dirId=3...,쳇 지피티 운세 어느정도 정확한가요? 무료로 꽤 정확한 운세 보는 방법 있으면 알려...
9,쳇지피티로 사주 해석을 해봤는데 맞는건가요?,https://kin.naver.com/qna/detail.naver?dirId=3...,쳇지피티로 배우자가 어떤 사람인지 사주 해석을 해봤는데 맞는건가요??? 안녕하세요 ...


In [2]:
# 방법1
import os
import requests
import json
import pandas as pd
client_id = os.getenv('CLIENT_ID')
client_secret = os.getenv('CLIENT_SECRET')
encText = "스위스"
url = f"https://openapi.naver.com/v1/search/kin.json" # JSON 결과
params = {'query':encText, 'display':20 }
headers = {
    'X-Naver-Client-Id': client_id,
    'X-Naver-Client-Secret':client_secret
}
response = requests.get(url, params=params, headers=headers)

items = response.json()['items'] # response를 json형태로 변환한것 중 'items'
items_list = []
for item in items:
    #print(item)
    title = item['title'].replace('<b>','').replace('</b>','')
    link  = item['link']
    description = item['description'].replace('<b>','').replace('</b>','')
    items_list.append([title, link, description])
pd.DataFrame(items_list, columns=['title','link','description']).sample()

,title,link,description
5,스위스의 지형,https://kin.naver.com/qna/detail.naver?dirId=8...,스위스의 지형은 국토의 절반 이상이 바위로 된 산이 많이 있나요? 스위스 지형 보면...


### 2) NAVER API HUB 방식으로 지식IN 검색

In [3]:
# 방법1
import os
import requests
import json
import pandas as pd
from dotenv import load_dotenv
load_dotenv() # 환경 변수 load
client_id = os.getenv('NEW_CLIENT_ID')
client_secret = os.getenv('NEW_CLIENT_SECRET')
encText = "쳇지피티"
url = f"https://naverapihub.apigw.ntruss.com/search/v1/kin" # JSON 결과
params = {'query':encText, 'display':20 }
headers = {
    'X-NCP-APIGW-API-KEY-ID': client_id,
    'X-NCP-APIGW-API-KEY':client_secret
}
response = requests.get(url, params=params, headers=headers)
items = response.json()['items'] # response를 json형태로 변환한것 중 'items'
items_list = []
for item in items:
    #print(item)
    title = item['title'].replace('<b>','').replace('</b>','')
    link  = item['link']
    description = item['description'].replace('<b>','').replace('</b>','')
    items_list.append([title, link, description])
pd.DataFrame(items_list, columns=['title','link','description']).sample()

,title,link,description
17,뉴스 보다가 범죄자가 검색한 쳇 지피티 도 조회 한다고 하....,https://kin.naver.com/qna/detail.naver?dirId=6...,제미나이 이런것도 포렌식 해서 다 검열을 하나요 첨 알아서요 검색 목록만 보는줄 알...


### quiz) 네이버 open API를 이용하여 원하는 query 이미지 100건의데이터를 'data/img_list.csv'파일로 
- title(제목), link(링크), thumbnail(썸네일), sizeheight, sizewidth

In [3]:
def get_image_list(query):
    'query로 검색한 이미지 정보(제목, 링크,썸네일, size) 100건 데이터 프레임을 return(방법1)'
    from dotenv import load_dotenv
    import os
    import requests
    import pandas as pd
    load_dotenv()
    client_id = os.getenv('CLIENT_ID')
    client_secret = os.getenv('CLIENT_SECRET')
    headers = {
        'X-Naver-Client-Id': client_id,
        'X-Naver-Client-Secret':client_secret
    }
    url = 'https://openapi.naver.com/v1/search/image'
    params = {'query':query, 'display':100 }
    response = requests.get(url, params=params, headers=headers)
    
    items = response.json()['items']
    # print(items[:2])
    items_list = []
    for item in items:
        items_list.append({
            '제목':item.get('title'),
            '링크':item.get('link'),
            '썸네일':item.get('thumbnail'),
            'sizeheight':item.get('sizeheight'),
            'sizewidth':item.get('sizewidth')
        })
    return pd.DataFrame(items_list)
get_image_list("청바지")    

,제목,링크,썸네일,sizeheight,sizewidth
0,[홍은]통바지 데님 연청 여자 하이웨이스트 와이드 청바지 | 텐바이텐,https://thumbnail.10x10.co.kr/webimage/image/b...,https://search.pstatic.net/sunny/?type=b150&sr...,500,500
1,여성용 와이드 하이웨스트 데님팬츠 데일리 청바지 5516,http://shopping.phinf.naver.net/main_5712970/5...,https://search.pstatic.net/common/?type=b150&s...,800,800
2,가을 겨울 일자핏 캐주얼 와이드 청바지 DNZK03 : 스마트 공장,http://shop1.phinf.naver.net/20241125_266/1732...,https://search.pstatic.net/common/?type=b150&s...,916,750
3,여자 스키니진 스판 타이트 청바지 | 텐바이텐,https://thumbnail.10x10.co.kr/webimage/image/b...,https://search.pstatic.net/sunny/?type=b150&sr...,500,500
4,남자 와이드 스랙스 9부 바지 가 남성 청바지 루즈한 스트레이트 리 광저우 : so...,http://shop1.phinf.naver.net/20260331_151/1774...,https://search.pstatic.net/common/?type=b150&s...,1000,750
...,...,...,...,...,...
95,MOSAIRATION 여성 청바지 하이웨스트 캐주얼 스트레이트 데님 팬츠 M0011214,http://shopping.phinf.naver.net/main_5905233/5...,https://search.pstatic.net/common/?type=b150&s...,1000,1000
96,여성 바지 빈티지 캐주얼 가을 청바지 데님바지 엘보 배기핏 와이드 워싱 데일룩 DA...,https://shop-phinf.pstatic.net/20260225_102/17...,https://search.pstatic.net/common/?type=b150&s...,500,500
97,여성 슬림핏 하이웨이스트 데님 청바지 봄 와이드 스트레이트 크롭 : 하유유통,https://shop-phinf.pstatic.net/20260605_15/178...,https://search.pstatic.net/common/?type=b150&s...,800,800
98,타미힐피거 기본 워싱 청바지 EMM3PD91A,http://shopping.phinf.naver.net/main_6623271/6...,https://search.pstatic.net/common/?type=b150&s...,500,500


In [34]:
def get_image_list(query):
    'query로 검색한 이미지 정보(제목, 링크,썸네일, size) 100건 데이터 프레임을 return(방법2)'
    from dotenv import load_dotenv
    import os
    from urllib.request import urlopen, Request
    from urllib.parse import quote
    import json
    import pandas as pd
    load_dotenv()
    client_id = os.getenv('CLIENT_ID')
    client_secret = os.getenv('CLIENT_SECRET')
    headers = {
        'X-Naver-Client-Id': client_id,
        'X-Naver-Client-Secret':client_secret
    }
    query = quote(query)
    url = f'https://openapi.naver.com/v1/search/image?query={query}&display=100'
    request = Request(url, headers=headers)
    response = urlopen(request)
    
    items = json.loads(response.read())['items']
    # print(items[:2])
    items_list = []
    for item in items:
        items_list.append({
            '제목':item.get('title'),
            '링크':item.get('link'),
            '썸네일':item.get('thumbnail'),
            'sizeheight':item.get('sizeheight'),
            'sizewidth':item.get('sizewidth')
        })
    return pd.DataFrame(items_list)
df = get_image_list("청바지")
df.sample()

,제목,링크,썸네일,sizeheight,sizewidth
99,TKX 남성 복고풍 자수 디테일 와이드 청바지 캐주얼 스타일 봄가을 워싱 편한 사진색,http://shopping.phinf.naver.net/main_5874184/5...,https://search.pstatic.net/common/?type=b150&s...,1000,1000


In [11]:
df.to_csv('data/img_list.csv', encoding='cp949', index=False)

In [12]:
import pandas as pd
df = pd.read_csv('data/img_list.csv', encoding='cp949')

In [17]:
# df에 있는 이미지 로컬에 저장하기
print(df.loc[0, '링크']) # 메인_01_청바지.jpg
print(df.loc[0, '썸네일']) # 썸네일_01_청바지.jpg

https://thumbnail.10x10.co.kr/webimage/image/basic600/741/B007415375.jpg?cmd=thumb&w=500&h=500&fit=true&ws=false
https://search.pstatic.net/sunny/?type=b150&src=https%3A%2F%2Fthumbnail.10x10.co.kr%2Fwebimage%2Fimage%2Fbasic600%2F741%2FB007415375.jpg%3Fcmd%3Dthumb%26w%3D500%26h%3D500%26fit%3Dtrue%26ws%3Dfalse


In [26]:
url = "https://a.net/a.jpg"
# '.'+url.split('.')[-1]
url[url.rfind('.'):]

'.jpg'

In [31]:
def save_image(attr, idx, link, query):
    'link의 이미지를 image/attr_idx_query.확장자로 local에 저장'
    import requests, os
    from urllib.parse import urlparse
    response = requests.get(link)
    # link에서 확장자(jpg) 추출
    file_extension = link.split('.')[-1]
    # 확장자 뒤에 ?가 있는 위치(?가 없으면 -1)
    index = file_extension.find('?') 
    if index != -1:
        file_extension = file_extension[:index]
    # 확장자 뒤에 %가 있는 위치
    index = file_extension.find('%')
    if index != -1:
        file_extension = file_extension[:index]
    # 허용할 이미지 확장자인지
    valid_extension = ['jpg', 'jpeg', 'png', 'gif', 'bmp', 'webp', 'svg']
    if file_extension.lower() not in valid_extension:
        # 허용할 이미지 확장자가 아닌 경우(ex: .net)
        print(f'{idx}번째 {attr}의 url에서 확장자를 추출 못하여 header에서 도전해봄')
        content_type = response.headers.get('Content-Type','') # image/jpeg
        file_extension = content_type.split('image/')[-1] # jpeg
        file_extension = file_extension.replace('jpeg', 'jpg')
        file_extension = file_extension if file_extension in valid_extension else 'jpg'
    # image 폴더가 없으면 image 폴더 생성
    save_dir = 'image'
    os.makedirs(save_dir, exist_ok=True)
    # 이미지 저장
    with open(f'{save_dir}/{attr}_{idx:02}_{query}.{file_extension}', 'wb') as f:
        f.write(response.content) # response의 바이너리를 저장
save_image('썸네일', 0, df.loc[0,'썸네일'], '청바지')

In [ ]:
# naver api 요청받아 이미지제목,link,썸네일link 정보를 csv로 백업, 이미지link와 썸네일link 이미지를 다운

In [36]:
def get_image_list_save_file(query):
    '''
    naver api 요청받아 이미지제목,link,썸네일link 정보를 데이터프레임으로 return
    데이터프레임을 csv로 백업
    이미지link와 썸네일link 이미지를 다운(방법2)
    '''
    from dotenv import load_dotenv
    import os
    from urllib.request import urlopen, Request
    from urllib.parse import quote
    import json
    import pandas as pd
    load_dotenv()
    client_id = os.getenv('CLIENT_ID')
    client_secret = os.getenv('CLIENT_SECRET')
    headers = {
        'X-Naver-Client-Id': client_id,
        'X-Naver-Client-Secret':client_secret
    }
    url = f'https://openapi.naver.com/v1/search/image?query={quote(query)}&display=100'
    request = Request(url, headers=headers)
    response = urlopen(request)
    
    items = json.loads(response.read())['items']
    items_list = []
    for idx, item in enumerate(items):
        link = item.get('link')
        thumbnail = item.get('thumbnail')
        items_list.append({
            '제목':item.get('title'),
            '링크':link,
            '썸네일':thumbnail,
            'sizeheight':item.get('sizeheight'),
            'sizewidth':item.get('sizewidth')
        })
        # 이미지 저장 save_image('메인', idx, link, query)  save_image('썸네일', idx, thumbnail, query)
        save_image('메인', idx, link, query)
        save_image('썸네일', idx, thumbnail, query)
        if idx%20==0:
            print(f'==={idx}% 진행완료 ===')
    result = pd.DataFrame(items_list)
    result.to_csv('image/img_list.csv', index=False)
    print('이미지 및 csv 저장완료')
    return result

In [38]:
df = get_image_list_save_file("청바지")

===0% 진행완료 ===
===20% 진행완료 ===
===40% 진행완료 ===
===60% 진행완료 ===
===80% 진행완료 ===
이미지 및 csv 저장완료


## 2.3 XML 웹 데이터 수집
- RSS서비스, openAPI사용
### 1) 전국 날씨 RSS를 BeautifulSoup을 이용한 xml크롤링
- 기상청 RSS에서 1개월 기상정보를 서비스

In [48]:
import requests
# from urllib.request import urlopen
from bs4 import BeautifulSoup
import pandas as pd
import numpy as np
items_list = []
url = 'https://www.kma.go.kr/repositary/xml/fct/mon/img/fct_mon1rss_108_20260813.xml'
target = requests.get(url)
soup = BeautifulSoup(target.text, 'xml')
locals = soup.select('local_ta')
#print(locals[1])
for local in locals:
    local_name = local.select_one('local_ta_name').text.strip()
    week1_local_ta_normalYear = local.select_one('week1_local_ta_normalYear').text # 평년기온
    week1_local_ta_similarRange= local.select_one('week1_local_ta_similarRange').text # 예측범위
    week1_local_ta_minVal      = local.select_one('week1_local_ta_minVal').text # 평년보다 낮을 확률
    week1_local_ta_similarVal  = local.select_one('week1_local_ta_similarVal').text # 평년과 비슷할 확률
    week1_local_ta_maxVal      = local.select_one('week1_local_ta_maxVal').text # 평년보다 높을 확률
    items_list.append({
        '지역': local_name,
        '평년기온':week1_local_ta_normalYear,
        '예측범위':week1_local_ta_similarRange,
        '낮을확률':week1_local_ta_minVal,
        '같을확률':week1_local_ta_similarVal,
        '높을확률':week1_local_ta_maxVal,
    })
df = pd.DataFrame(items_list)
df.head()

,지역,평년기온,예측범위,낮을확률,같을확률,높을확률
0,"전국(제주도,북한제외)",23.7,23.1~24.3,10,30,60
1,서울ㆍ인천ㆍ경기도,23.9,23.3~24.5,10,30,60
2,강원도 영서,22.1,21.4~22.8,10,30,60
3,강원도 영동,22.0,21.3~22.7,10,30,60
4,대전ㆍ세종ㆍ충청남도,24.0,23.4~24.6,10,30,60


In [49]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13 entries, 0 to 12
Data columns (total 6 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   지역      13 non-null     object
 1   평년기온    13 non-null     object
 2   예측범위    13 non-null     object
 3   낮을확률    13 non-null     object
 4   같을확률    13 non-null     object
 5   높을확률    13 non-null     object
dtypes: object(6)
memory usage: 752.0+ bytes


In [52]:
df['평년기온'] = df['평년기온'].astype(np.float64)
df['낮을확률'] = df['낮을확률'].astype(np.int16)
df['같을확률'] = df['같을확률'].astype(np.int16)
df['높을확률'] = df['높을확률'].astype('int')
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13 entries, 0 to 12
Data columns (total 6 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   지역      13 non-null     object 
 1   평년기온    13 non-null     float64
 2   예측범위    13 non-null     object 
 3   낮을확률    13 non-null     int16  
 4   같을확률    13 non-null     int16  
 5   높을확률    13 non-null     int32  
dtypes: float64(1), int16(2), int32(1), object(2)
memory usage: 544.0+ bytes


In [53]:
df

,지역,평년기온,예측범위,낮을확률,같을확률,높을확률
0,"전국(제주도,북한제외)",23.7,23.1~24.3,10,30,60
1,서울ㆍ인천ㆍ경기도,23.9,23.3~24.5,10,30,60
2,강원도 영서,22.1,21.4~22.8,10,30,60
3,강원도 영동,22.0,21.3~22.7,10,30,60
4,대전ㆍ세종ㆍ충청남도,24.0,23.4~24.6,10,30,60
5,충청북도,23.2,22.6~23.8,10,30,60
6,광주ㆍ전라남도,24.9,24.3~25.5,10,30,60
7,전북자치도,23.9,23.3~24.5,10,30,60
8,부산ㆍ울산ㆍ경상남도,24.6,24.0~25.2,10,30,60
9,대구ㆍ경상북도,23.4,22.8~24.0,10,30,60


### 3) xml 응답하는 Open API 활용
- data.go.kr에서 
    * 서울특별시_노선정보조회 서비스 활용 신청(버스 id, 버스 정류장 목록)
    * 버스 위치 정보 조회 서비스 활용 신청(실시간 버스 위치 목록)

In [35]:
# step1. 버스 번호의  busRouteId 받아오기
# 서울특별시_노선정보조회 서비스 - 3번 기능(getBusRouteList) 이용
# http://api.bus.go.kr/contents/sub02/getBusRouteList.html

In [4]:
from dotenv import load_dotenv
import os
load_dotenv()

True

In [47]:
import requests
from bs4 import BeautifulSoup
from urllib.request import urlretrieve #
from urllib.parse import quote
# busNum = '마포01'
busNum = '162'
key = os.getenv('key')
# url1 = f'http://ws.bus.go.kr/api/rest/busRouteInfo/getBusRouteList?strSrch={quote(busNum)}&ServiceKey={key}'
url1 = f'http://ws.bus.go.kr/api/rest/busRouteInfo/getBusRouteList?strSrch={busNum}&ServiceKey={key}'
print(url1)
# savefilename1 = 'data/ch14_1.busInfo.xml'
# urlretrieve(url1, savefilename1)
# with open(savefilename1, encoding='utf-8') as f:
#     xml = f.read();
# soup = BeautifulSoup(xml, 'xml')
xml = requests.get(url1)
soup = BeautifulSoup(xml.text, 'xml')
#soup

http://ws.bus.go.kr/api/rest/busRouteInfo/getBusRouteList?strSrch=162&ServiceKey=80cf6cae1b938a0afcb1b250b1de989adf70e09e17611ca3481e8be6881718af


In [48]:
for item in soup.select('itemList'):
    busRouteNm = item.select_one('busRouteNm').text
    if busNum == busRouteNm:
        busRouteId = item.select_one('busRouteId').text
        break
print('busRouteId =', busRouteId)

busRouteId = 100100034


In [ ]:
# step2. 해당 busRouteId의  경유 정류장 목록 받아오기(정류장id, 정류장이름)
# 서울특별시_노선정보조회 서비스 - 4번 기능(getStaionsByRouteList) 이용
# http://api.bus.go.kr/contents/sub02/getStaionByRoute.html

In [49]:
import pandas as pd
url2 = f'http://ws.bus.go.kr/api/rest/busRouteInfo/getStaionByRoute?ServiceKey={key}&busRouteId={busRouteId}'
print(url2)
response = requests.get(url2)
soup = BeautifulSoup(response.text, 'xml')
itemLists = soup.select('itemList')
print(f'{busNum}번 정류장 갯수 :', len(itemLists))
bus_station = []
for itemList in itemLists:
    stationNm = itemList.select_one('stationNm').text # 정류장명
    station   = itemList.select_one('station').text   # 정류장ID
    gpsX      = itemList.select_one('gpsX').text # 경도
    gpsY      = itemList.select_one('gpsY').text # 위도
    bus_station.append([stationNm, station, gpsX, gpsY])
df_station = pd.DataFrame(bus_station, columns=['정류소명','id','경도','위도'])
df_station

http://ws.bus.go.kr/api/rest/busRouteInfo/getStaionByRoute?ServiceKey=80cf6cae1b938a0afcb1b250b1de989adf70e09e17611ca3481e8be6881718af&busRouteId=100100034
162번 정류장 갯수 : 77


,정류소명,id,경도,위도
0,정릉산장아파트,107000071,127.003343,37.616712
1,정릉4동주민센터.경국사,107000073,127.006345,37.613529
2,북한산보국문역2번출구,107000518,127.0079858233,37.612293899
3,성북청수도서관.정릉4동성당,107000075,127.0084193769,37.6115696748
4,정릉시장입구,107000077,127.0098212542,37.6084653256
...,...,...,...,...
72,성북청수도서관.정릉4동성당,107000076,127.009045,37.610876
73,북한산보국문역1번출구,107000519,127.008329146,37.6120835499
74,정릉4동주민센터.경국사,107000074,127.006681,37.613335
75,정릉대우아파트,107000072,127.00386,37.616708


In [44]:
pd.options.display.max_rows

60

In [ ]:
# step3. 해당 busRouteId의 차량들의 위치정보(차량번호, 혼잡도, 경도, 위도, 최종정류장id, 다음정류장id, 도착소요시간)
# 서울특별시_버스위치정보조회 서비스 - 2번 기능(getBusPosByRtidList) 이용
# http://api.bus.go.kr/contents/sub02/getBusPosByRtid.html

In [84]:
url3 = f'http://ws.bus.go.kr/api/rest/buspos/getBusPosByRtid?serviceKey={key}&busRouteId={busRouteId}'
print(url3)
response = requests.get(url3)
soup = BeautifulSoup(response.text, 'xml')
itemLists = soup.select('itemList')
print(f'{busNum}번 운행중인 버스는 {len(itemLists)}대입니다')
bus_position = [] # 버스 위치정보를 담을 list
for itemList in itemLists:
    plainNo = itemList.select_one('plainNo').text # 차량번호
    congetion = itemList.select_one('congetion').text 
    #0:없음, 3:여유, 4:보통, 5:혼잡
    congetion = '없음' if congetion=='0' \
            else '여유' if congetion=='3' \
            else '보통' if congetion=='4' \
            else '혼잡'
    gpsX = itemList.select_one('gpsX').text # 경도
    gpsY = itemList.select_one('gpsY').text # 위도
    lastStnId = itemList.select_one('lastStnId').text # 최종정류소id
    nextStId  = itemList.select_one('nextStId').text # 다음정류소id
    nextStTm  = itemList.select_one('nextStTm').text # 다음정류소도착소요시간
    bus_position.append({
        '차량번호': plainNo,
        '혼잡도':congetion,
        '경도':gpsX,
        '위도':gpsY,
        '최종정류소id':lastStnId,
        '다음정류소id':nextStId,
        '도착소요시간':nextStTm
    })
df_position = pd.DataFrame(bus_position)
df_position.head(2)

http://ws.bus.go.kr/api/rest/buspos/getBusPosByRtid?serviceKey=80cf6cae1b938a0afcb1b250b1de989adf70e09e17611ca3481e8be6881718af&busRouteId=100100034
162번 운행중인 버스는 23대입니다


,차량번호,혼잡도,경도,위도,최종정류소id,다음정류소id,도착소요시간
0,서울70사6553,없음,127.001808,37.617234,107000071,107000169,311
1,서울70사6560,여유,127.007986,37.612294,107000518,107000079,229


In [85]:
df_station.loc[df_station['id']=='107000518','정류소명'].iloc[0]

'북한산보국문역2번출구'

In [98]:
def get_station_name(row):
    row['최종정류소명'] = df_station.loc[df_station['id']==row['최종정류소id'], '정류소명'].iloc[0]
    row['다음정류소명'] = df_station.loc[df_station['id']==row['다음정류소id'], '정류소명'].iloc[0]
    return row

In [108]:
#get_station_name(df_position.iloc[0])
df_position = df_position.apply(get_station_name, axis=1)
df_position.head(1)

,차량번호,혼잡도,경도,위도,최종정류소id,다음정류소id,도착소요시간,최종정류소명,다음정류소명
0,서울70사6553,없음,127.001808,37.617234,107000071,107000169,311,정릉산장아파트,정릉입구.정릉역


In [110]:
drop_col = df_position.columns.str.contains('id')
drop_column_names = df_position.columns[drop_col]
df_position.drop(drop_column_names, axis=1, inplace=True)

In [118]:
df_position['도착소요시간'] = round( df_position['도착소요시간'].astype('int')/60, 2 )
df_position.head()

,차량번호,혼잡도,경도,위도,도착소요시간,최종정류소명,다음정류소명
0,서울70사6553,없음,127.001808,37.617234,5.18,정릉산장아파트,정릉입구.정릉역
1,서울70사6560,여유,127.007986,37.612294,3.82,북한산보국문역2번출구,정릉우체국앞
2,서울74사3360,여유,127.013778,37.600632,34.32,아리랑고개.아리랑시네미디어센터,소공동.롯데영플라자
3,서울74사1625,여유,127.016325,37.593333,30.88,성신여대입구역5번출구,소공동.롯데영플라자
4,서울74사2216,여유,127.009335,37.589965,27.32,삼선교.한성대학교,소공동.롯데영플라자


# 3절. 연습문제

- YES24 베스트셀러 정보에서 순위1~48위까지의 ***[순위, 책이름, 저장, 출판사, 가격]***정보를 출력하고, ch14_yes24_bestseller.csv로 백업

In [119]:
# import
import requests
from urllib.request import urlopen
from bs4 import BeautifulSoup
import pandas as pd
import re

In [142]:
pages = 2
bestseller_list = []
for page in range(1, pages+1):
    url = f'https://www.yes24.com/product/category/bestseller?categoryNumber=001&pageNumber={page}&pageSize=24'
    # 방법1
    response = requests.get(url)
    soup = BeautifulSoup(response.text, 'html.parser')
    # 방법2
#     response = urlopen(url)
#     soup = BeautifulSoup(response, 'html.parser')
    # 순위(48개)
    ranks_els = soup.select('div.img_upper > em.ico.rank')
    ranks = [int(ranks_el.text) for ranks_el in ranks_els]
    # 책제목(48개)
    titles_els = soup.select('div.info_row.info_name > a.gd_name')
    titles = [titles_el.text for titles_el in titles_els]
    # 저자(a태그까지 하면 저자가 48명이 아니고 더 많아짐. 공저나 변역가, 그림이 있어서)
    authors_els = soup.select('div.info_row.info_pubGrp > span.authPub.info_auth')
    authors = [authors_el.text.strip() for authors_el in authors_els]
    # 출판사(48개)
    publishers_els = soup.select('div.info_row.info_pubGrp > span.authPub.info_pub')
    publishers = [publishers_el.text for publishers_el in publishers_els]
    # 가격(48개)
    prices_els = soup.select('div.info_row.info_price > strong.txt_num')
    prices = [prices_el.text for prices_el in prices_els]
    for rank, title, author, publisher, price in zip(ranks, titles, authors, publishers, prices):
        bestseller_list.append([rank, title, author, publisher, price])
    # time.sleep(2) : https://www.yes24.com/robots.txt에 해당 url을 막지 않아서 time.sleep안 함
df = pd.DataFrame(bestseller_list, columns=['순위','책이름','저자','출판사','가격'])
df.to_csv('data/ch14_yes24_bestseller.csv', index=False)
df.head()

,순위,책이름,저자,출판사,가격
0,1,"세네카, 오늘을 빼앗기고 있는 당신에게",루키우스 안나이우스 세네카 저/하와이 대저택 편역,논픽션,"16,200원"
1,2,오디세이아,호메로스 저/페테르 파울 루벤스 그림/박문재 역,현대지성,"24,300원"
2,3,원소 원정대: 118개 캐릭터로 마스터하는 주기율표 공략집,아게도리도리 글그림/박재현 역/장홍제 감수,윌북주니어,"17,820원"
3,4,오뒷세이아,호메로스 저/김헌 역,을유문화사,"22,500원"
4,5,싯다르타,헤르만 헤세 저/박병덕 역,민음사,"7,200원"
